# 3.13 Spillover exposure stats

Descriptive stats for the paper re. spillovers analysis. Sample: the 138 lab groups surveyed at baseline.

In [1]:
# Set-up
import pandas as pd
import sys
from pathlib import Path
from itertools import combinations
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

In [2]:
# Load data
df = pd.read_csv(config.CLEAN_DATA / "final_dataset.csv", keep_default_na=False, na_values=[""])

share_equip_groups = pd.read_csv(config.CLEAN_DATA / "share_equip_groups_cleaned.csv", keep_default_na=False, na_values=[""])
share_space_groups = pd.read_csv(config.CLEAN_DATA / "share_space_groups_cleaned.csv", keep_default_na=False, na_values=[""])
publications = pd.read_csv(config.PUBLICATON_DATA / "2_Processed" / "publications_matched.csv")

In [3]:
# Restrict to the BL sample: every lab group surveyed at baseline has a treatment assignment,
# regardless of whether they later completed EL too
bl = df[df["survey"] == "BL"].drop_duplicates("labgroupid")
treated_by_lab = bl.set_index("labgroupid")["treated"]

n_sample = len(treated_by_lab)
print(f"{n_sample} lab groups in the sample")

138 lab groups in the sample


## (1) Cross-arm equipment and space sharing

In [4]:
# Share of sample labs that report sharing equipment/space with >=1 group in the opposite arm.
# Self-referencing rows and partners outside the sample (unknown treatment status) are dropped -
# a lab with no valid ties simply counts as not having a cross-arm tie, not missing.
def share_cross_arm(sharing_df):
    d = sharing_df.dropna(subset=["sample_group"]).copy()
    d["sample_group"] = d["sample_group"].astype(int)
    d = d[d["labgroupid"] != d["sample_group"]]
    d = d[d["labgroupid"].isin(treated_by_lab.index) & d["sample_group"].isin(treated_by_lab.index)]

    d["own_treated"] = d["labgroupid"].map(treated_by_lab)
    d["partner_treated"] = d["sample_group"].map(treated_by_lab)
    cross_arm_labs = set(d.loc[d["own_treated"] != d["partner_treated"], "labgroupid"])

    return len(cross_arm_labs) / n_sample, len(cross_arm_labs)

share_equip_cross, n_equip_cross = share_cross_arm(share_equip_groups)
share_space_cross, n_space_cross = share_cross_arm(share_space_groups)

print(f"Equipment sharing, cross-arm: {n_equip_cross} of {n_sample} labs ({share_equip_cross:.1%})")
print(f"Space sharing, cross-arm: {n_space_cross} of {n_sample} labs ({share_space_cross:.1%})")

Equipment sharing, cross-arm: 48 of 138 labs (34.8%)
Space sharing, cross-arm: 39 of 138 labs (28.3%)


## (2) Publication ties

In [5]:
# Share of consenting sample labs with >=1 joint (2+ labgroupid) publication with another consenting lab group. 
bl_indexed = bl.set_index("labgroupid")
consenting = bl_indexed.index[bl_indexed["consent_data_merge"] == "Yes I consent to this data collection and merging"]
n_consenting = len(consenting)

multi_lab_pubs = publications[publications["n_matched_labgroupids"] > 1]
linked_labs = set()
for ids in multi_lab_pubs["matched_labgroupids"]:
    linked_labs.update(int(x) for x in str(ids).split(";"))
linked_consenting_labs = linked_labs & set(consenting)

share_pub_tie = len(linked_consenting_labs) / n_consenting
print(f"Publication ties: {len(linked_consenting_labs)} of {n_consenting} consenting labs ({share_pub_tie:.1%})")

Publication ties: 57 of 104 consenting labs (54.8%)


In [6]:
# Share of consenting labs with >=1 co-authored publication with a group in the opposite arm
# (same "cross-arm" definition as equipment/space sharing above)
pub_edges = set()
for ids in multi_lab_pubs["matched_labgroupids"]:
    lab_ids = sorted(set(int(x) for x in str(ids).split(";")))
    pub_edges.update(combinations(lab_ids, 2))

pub_edges = pd.DataFrame(pub_edges, columns=["labgroupid_a", "labgroupid_b"])
pub_edges = pub_edges[pub_edges["labgroupid_a"].isin(consenting) & pub_edges["labgroupid_b"].isin(consenting)]

pub_edges["treated_a"] = pub_edges["labgroupid_a"].map(treated_by_lab)
pub_edges["treated_b"] = pub_edges["labgroupid_b"].map(treated_by_lab)
cross = pub_edges[pub_edges["treated_a"] != pub_edges["treated_b"]]
cross_arm_pub_labs = set(cross["labgroupid_a"]) | set(cross["labgroupid_b"])

share_pub_cross = len(cross_arm_pub_labs) / n_consenting
print(f"Publication ties, cross-arm: {len(cross_arm_pub_labs)} of {n_consenting} consenting labs ({share_pub_cross:.1%})")

Publication ties, cross-arm: 46 of 104 consenting labs (44.2%)
